# Second-order plant, then PID feedback
Run the two code cells in order, then move the sliders. Plots update when you release a slider. Plant sliders in the first section also update the PID comparison below. If you rerun the first cell, rerun the second to reconnect it.

Requires NumPy, SciPy, Matplotlib, and **ipywidgets** in the selected kernel. If widgets are missing, run `%pip install ipywidgets` in a temporary notebook cell, then restart the kernel.

We use a stable, delay-free plant: $G(s)=K\omega_n^2/(s^2+2\zeta\omega_n s+\omega_n^2)$. For a series RLC circuit with capacitor-voltage output, $K=1$, $\omega_n=1/\sqrt{LC}$, and $\zeta=(R/2)\sqrt{C/L}$. All responses start from rest.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import ipywidgets as widgets
from IPython.display import display
from io import BytesIO, StringIO
from contextlib import redirect_stdout
from html import escape

# Disconnect callbacks explicitly; closing an Output alone leaves observers attached.
for binding in globals().get('_plot_bindings', {}).values():
    for control in binding['controls'].values():
        control.unobserve(binding['callback'], names='value')
    binding['output'].close()
_plot_bindings = {}

def disconnect_plot(name):
    binding = _plot_bindings.pop(name, None)
    if binding is not None:
        for control in binding['controls'].values():
            control.unobserve(binding['callback'], names='value')
        binding['output'].close()

def bind_plot(name, function, controls):
    disconnect_plot(name)
    image = widgets.Image(format='png', layout=widgets.Layout(max_width='100%'))
    details = widgets.HTML()
    output = widgets.VBox([image, details])
    def redraw(change=None):
        # Replace bytes in one existing image; never append notebook plot outputs.
        text = StringIO()
        with redirect_stdout(text), plt.ioff():
            fig = function(**{key: control.value for key, control in controls.items()})
            buffer = BytesIO()
            try:
                fig.savefig(buffer, format='png', dpi=100, bbox_inches='tight')
                image.value = buffer.getvalue()
            finally:
                plt.close(fig)
                buffer.close()
        details.value = '<pre>' + escape(text.getvalue()) + '</pre>'
    for control in controls.values():
        control.observe(redraw, names='value')
    _plot_bindings[name] = {
        'controls': controls, 'callback': redraw, 'output': output,
    }
    redraw()
    return output

def slider(label, value, lower, upper, step):
    return widgets.FloatSlider(
        description=label, value=value, min=lower, max=upper, step=step,
        continuous_update=False, style={'description_width': '150px'},
        layout=widgets.Layout(width='550px'), readout_format='.2f')

plant_controls = {
    'zeta': slider('Damping zeta', 0.2, 0.05, 2.0, 0.05),
    'wn': slider('Frequency wn (rad/s)', 1.0, 0.2, 10.0, 0.1),
    'K': slider('Plant gain K', 1.0, 0.1, 3.0, 0.1),
    'setpoint': slider('Setpoint', 1.0, 0.1, 5.0, 0.1),
    'duration': slider('Duration (s)', 30.0, 5.0, 100.0, 1.0),
}

def plot_plant(zeta, wn, K, setpoint, duration):
    assert zeta > 0 and wn > 0 and K > 0 and duration > 0
    assert setpoint > 0, 'Use a positive setpoint for this experiment.'
    t = np.linspace(0, duration, 3000)
    
    # G(s): numerator and denominator coefficients in descending powers of s.
    plant_num = np.array([K * wn**2])
    plant_den = np.array([1.0, 2 * zeta * wn, wn**2])
    plant = signal.TransferFunction(plant_num, plant_den)
    _, unit_response = signal.step(plant, T=t)
    y_open = setpoint * unit_response
    
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(t, y_open, label='Plant alone: constant input = setpoint')
    ax.axhline(setpoint, color='black', linestyle='--', label='Desired output')
    ax.set(title='Open-loop step response', xlabel='Time (s)', ylabel='Output y(t)')
    ax.grid(alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.close(fig)  # Prevent automatic inline display from duplicating the figure.

    print(f'Open-loop final output approaches K Ã— input = {K * setpoint:g}.')
    print('No controller is present: the input stays constant regardless of the output.')
    return fig

plant_output = bind_plot('plant', plot_plant, plant_controls)
display(widgets.VBox(list(plant_controls.values())), plant_output)


## Add unity negative PID feedback
The error is $e=r-y$. The controller adjusts the plant input using $u=C(s)e$, giving $Y/R=CG/(1+CG)$ and $U/R=C/(1+CG)$.

Use a filtered derivative: $C(s)=K_p+K_i/s+K_ds/(T_fs+1)$. The filter avoids an ideal derivative impulse when the setpoint jumps. There are no actuator limits or automatic gain tuning. This filtered controller differs from the ideal PID in the dataset generator.

Try P only (Ki = Kd = 0), then add integral action, then derivative action using the sliders. Observe tracking error, overshoot, and control effort.


In [ ]:
# Run the plant cell first. Its sliders also update these feedback plots.
disconnect_plot('pid')
pid_controls = {
    'Kp': slider('Kp', 1.0, 0.0, 20.0, 0.1),
    'Ki': slider('Ki', 0.5, 0.0, 20.0, 0.1),
    'Kd': slider('Kd', 0.5, 0.0, 10.0, 0.1),
    'Tf': slider('Derivative filter (s)', 0.05, 0.01, 0.5, 0.01),
}

def plot_feedback(zeta, wn, K, setpoint, duration, Kp, Ki, Kd, Tf):
    t = np.linspace(0, duration, 3000)
    plant_num = np.array([K * wn**2])
    plant_den = np.array([1.0, 2 * zeta * wn, wn**2])
    _, unit_response = signal.step((plant_num, plant_den), T=t)
    y_open = setpoint * unit_response
    assert min(Kp, Ki, Kd) >= 0 and Tf > 0
    
    # Build C(s); omit the integrator when Ki is zero.
    if Ki == 0:
        c_num = np.trim_zeros([Kp * Tf + Kd, Kp], 'f')
        c_den = np.array([Tf, 1.0])
    else:
        c_num = np.trim_zeros([Kp * Tf + Kd, Kp + Ki * Tf, Ki], 'f')
        c_den = np.array([Tf, 1.0, 0.0])
    
    if len(c_num) == 0:  # All gains zero: no input and no output.
        closed_den = plant_den
        y_pid = np.zeros_like(t)
        u_pid = np.zeros_like(t)
    else:
        loop_num = np.polymul(c_num, plant_num)
        loop_den = np.polymul(c_den, plant_den)
        closed_den = np.polyadd(loop_den, loop_num)
        _, y_unit = signal.step((loop_num, closed_den), T=t)
        _, u_unit = signal.step((np.polymul(c_num, plant_den), closed_den), T=t)
        y_pid = setpoint * y_unit
        u_pid = setpoint * u_unit
    
    poles = np.roots(closed_den)
    stable = bool(np.all(poles.real < 0))
    print('Closed-loop poles:', poles)
    print('Asymptotically stable:', stable)
    
    fig, (ax_y, ax_u) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
    ax_y.plot(t, y_open, label='Plant alone')
    ax_y.axhline(setpoint, color='black', linestyle='--', label='Setpoint')
    if stable and np.all(np.isfinite(y_pid)) and np.all(np.isfinite(u_pid)):
        ax_y.plot(t, y_pid, label='Plant with PID feedback')
        ax_u.plot(t, u_pid, color='tab:purple', label='PID control input')
        overshoot = max(0.0, (np.max(y_pid) / setpoint - 1) * 100)
        print(f'Sampled overshoot: {overshoot:.2f}%')
        print(f'Error at end of simulation: {setpoint - y_pid[-1]:.5g}')
    else:
        print('PID traces hidden: unstable/marginal or nonfinite response. Adjust gains.')
    ax_u.axhline(setpoint, color='gray', linestyle=':', label='Constant input without PID')
    ax_y.set(title='Open loop versus PID feedback', ylabel='Output y(t)')
    ax_u.set(xlabel='Time (s)', ylabel='Control input u(t)')
    for ax in (ax_y, ax_u):
        ax.grid(alpha=0.3)
        ax.legend()
    plt.tight_layout()
    plt.close(fig)  # Display exactly once, inside this plot output.

    print('End error is measured at the final time, not necessarily at steady state.')
    return fig

pid_output = bind_plot('pid', plot_feedback, {**plant_controls, **pid_controls})
display(widgets.VBox(list(pid_controls.values())), pid_output)
